# Sistem Rekomendasi Paket Internet — CRISP-DM
Notebook ini menjalankan tahap **Data Understanding (3.3.2)**, **Data Preparation (3.3.3)**, **Modeling (3.3.4)**, **Evaluasi (3.3.5)**, **Validasi Sistem Tambahan (Cronbach's Alpha)**, **Perbandingan Metode Tambahan (Content-Based Filtering)**, dan **Deployment (3.3.6)** persis mengikuti nomor sub-bab di draft skripsi. Tiap cell = satu sub-bab, outputnya langsung tampil di bawah cell supaya bisa langsung di-screenshot untuk Bab IV.

Logika di setiap cell memanggil fungsi ASLI dari `data_preparation.py` dan `recommender.py` (bukan ditulis ulang di sini), supaya notebook ini **tidak pernah beda hasil** dengan aplikasi Streamlit (`app.py` / `app_final.py`) yang dipakai untuk demo Deployment.

In [1]:
import sys
import pandas as pd
from IPython.display import display

sys.path.insert(0, r"D:\Skripsi_Rekomendasi_ACS")
import data_preparation as dp
import recommender as rec

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
pd.set_option("display.precision", 4)  # kolom angka jadi 4 digit, tidak melebar

## Load Data

In [2]:
print("="*70)
print("LOAD DATA")
print("="*70)

df_raw = pd.read_csv("usage_summary asli.csv")
df_raw.columns = df_raw.columns.str.strip()

katalog = pd.read_csv("Jenis_Layanan.csv")
katalog.columns = katalog.columns.str.strip()

print(f"usage_summary asli.csv  -> {df_raw.shape[0]} baris, {df_raw.shape[1]} kolom")
print(f"Jenis_Layanan.csv       -> {katalog.shape[0]} baris, {katalog.shape[1]} kolom")

LOAD DATA


usage_summary asli.csv  -> 644 baris, 27 kolom
Jenis_Layanan.csv       -> 22 baris, 4 kolom


## 3.3.2 Data Understanding

In [3]:
print("="*70)
print("3.3.2  DATA UNDERSTANDING")
print("="*70)

print(f"Jumlah Baris : {df_raw.shape[0]}")
print(f"Jumlah Kolom : {df_raw.shape[1]}")

print("\nDaftar Kolom Dataset:")
print(", ".join(df_raw.columns))

print("\nTipe Data (ringkasan):")
print(df_raw.dtypes.value_counts().to_string())

preview_cols = [
    "Username", "Alamat", "Paket Internet",
    "Total Waktu Online", "Total Online (detik)",
    "Total Pemakaian Kuota", "Total Kuota (bytes)",
]
print("\nContoh Data (kolom yang relevan untuk Feature Selection):")
display(df_raw[preview_cols].head(3))

missing = df_raw.isnull().sum()
missing = missing[missing > 0]
print("\nKolom dengan Missing Value:")
display(missing)

print("\nJumlah Data Duplikat:", df_raw.duplicated().sum())

3.3.2  DATA UNDERSTANDING
Jumlah Baris : 644
Jumlah Kolom : 27

Daftar Kolom Dataset:
ID Pelanggan, Username, Nama Depan, Nama Belakang, Email, No. Telepon, Alamat, Cabang, Paket Internet, Status, Tanggal Dipasang, Pembayaran Terakhir, Tanggal Kedaluwarsa, Kedaluwarsa, Metode Koneksi, Sedang Online, Total Waktu Online, Total Online (detik), Total Pemakaian Kuota, Total Kuota (bytes), Jumlah Sesi, Jumlah Kenaikan Paket, Jumlah Penurunan Paket, Jumlah Ganti Paket, Paket Awal, Paket Sekarang, Error

Tipe Data (ringkasan):
str        19
int64       5
float64     3

Contoh Data (kolom yang relevan untuk Feature Selection):


,Username,Alamat,Paket Internet,Total Waktu Online,Total Online (detik),Total Pemakaian Kuota,Total Kuota (bytes)
0,pws@gar3sukirman13,PERUM PWS JL.GARUDA 3 BLOK AJF16 NO 13 RT 002 ...,ACS Lite 165,5 hari 1 jam 37 menit,437860,79.77 GB,8.5648e+10
1,mgs@rt02nobon03,Percetakan dan Duplikat kunci (CsC).jln margas...,ACS Home 165,17 hari 9 jam 30 menit,1503050,32.17 GB,3.4541e+10
2,tr2@ea7ari11,"Triraksa Village 2, blok EA. 7 No. 11",ACS Home 165,9 hari 22 jam 49 menit,859753,79.46 GB,8.5316e+10



Kolom dengan Missing Value:


Nama Belakang          215
Alamat                   1
Pembayaran Terakhir     66
Tanggal Kedaluwarsa     41
Error                  644
dtype: int64


Jumlah Data Duplikat: 0


## 3.3.3 Data Preparation

### 3.3.3.1 Feature Selection

In [4]:
print("="*70)
print("3.3.3.1  FEATURE SELECTION")
print("="*70)

df_selected = dp.select_features(df_raw)

print(f"Atribut yang dipertahankan ada {len(df_selected.columns)}, yaitu:")
for i, col in enumerate(df_selected.columns, start=1):
    print(f"  {i}. {col}")

print(f"\nJumlah Baris : {df_selected.shape[0]}")
print(f"Jumlah Kolom : {df_selected.shape[1]}")

print("\nContoh Data:")
display(df_selected.head(3))

3.3.3.1  FEATURE SELECTION
Atribut yang dipertahankan ada 5, yaitu:
  1. Username
  2. Paket Internet
  3. Total Online (detik)
  4. Total Kuota (bytes)
  5. Alamat

Jumlah Baris : 644
Jumlah Kolom : 5

Contoh Data:


,Username,Paket Internet,Total Online (detik),Total Kuota (bytes),Alamat
0,pws@gar3sukirman13,ACS Lite 165,437860,8.5648e+10,PERUM PWS JL.GARUDA 3 BLOK AJF16 NO 13 RT 002 ...
1,mgs@rt02nobon03,ACS Home 165,1503050,3.4541e+10,Percetakan dan Duplikat kunci (CsC).jln margas...
2,tr2@ea7ari11,ACS Home 165,859753,8.5316e+10,"Triraksa Village 2, blok EA. 7 No. 11"


### 3.3.3.2 Data Cleaning

In [5]:
print("="*70)
print("3.3.3.2  DATA CLEANING")
print("="*70)

print("Jumlah Missing Value per Kolom (sebelum cleaning):")
display(df_selected.isnull().sum())

print(f"\nJumlah Data Duplikat (baris identik): {df_selected.duplicated().sum()}")

df_clean = dp.clean_missing(df_selected)

print(f"\nJumlah Baris sebelum cleaning : {df_selected.shape[0]}")
print(f"Jumlah Baris setelah cleaning : {df_clean.shape[0]}")
print("(baris berkurang karena missing value dibuang, lalu duplikat "
      "Username+Paket Internet diagregasi jadi satu baris)")

print("\nContoh Data Setelah Cleaning:")
display(df_clean.head(3))

3.3.3.2  DATA CLEANING
Jumlah Missing Value per Kolom (sebelum cleaning):


Username                0
Paket Internet          0
Total Online (detik)    0
Total Kuota (bytes)     0
Alamat                  1
dtype: int64


Jumlah Data Duplikat (baris identik): 0



Jumlah Baris sebelum cleaning : 644


Jumlah Baris setelah cleaning : 643
(baris berkurang karena missing value dibuang, lalu duplikat Username+Paket Internet diagregasi jadi satu baris)

Contoh Data Setelah Cleaning:


,Username,Paket Internet,Total Online (detik),Total Kuota (bytes),Alamat
0,DANAN,WARGANET Lite 150,7488118,4.0300e+11,Sudirman B9 no 09
1,ELFA,WARGANET Lite 100,0,0.0000e+00,B7/38 sudirman
2,POSBLOKA,FREE 8U,11115746,5.8700e+11,POS BOLK A


### 3.3.3.3 Data Parsing (Pembuktian Persamaan 6)

In [6]:
print("="*70)
print("3.3.3.3  DATA PARSING (PEMBUKTIAN PERSAMAAN 6)")
print("="*70)

verify_df = dp.verify_parsing(df_raw)

if verify_df.empty:
    print("Kolom teks mentah tidak ditemukan, verifikasi dilewati.")
else:
    n_konsisten = int(verify_df["Konsisten"].sum())
    print(f"Baris konsisten (parser teks vs kolom numerik) : {n_konsisten}/{len(verify_df)}")
    print("\nContoh Hasil Parsing:")
    display(verify_df.head(3))

3.3.3.3  DATA PARSING (PEMBUKTIAN PERSAMAAN 6)


Baris konsisten (parser teks vs kolom numerik) : 644/644

Contoh Hasil Parsing:


,Username,Online (detik) - parsed,Online (detik) - asli,Selisih relatif Online,Kuota (bytes) - parsed,Kuota (bytes) - asli,Selisih relatif Kuota,Konsisten
0,pws@gar3sukirman13,437820.0,437860,0.0001,8.5652e+10,8.5648e+10,0.0001,True
1,mgs@rt02nobon03,1503000.0,1503050,0.0,3.4542e+10,3.4541e+10,0.0,True
2,tr2@ea7ari11,859740.0,859753,0.0,8.5320e+10,8.5316e+10,0.0,True


### 3.3.3.4 Data Integration (Data Mapping + Inner Join)

In [7]:
print("="*70)
print("3.3.3.4  DATA INTEGRATION (DATA MAPPING + INNER JOIN)")
print("="*70)

df_integrated = dp.integrate_catalog(df_clean, katalog)

print(f"\nJumlah Baris sebelum integrasi : {df_clean.shape[0]}")
print(f"Jumlah Baris setelah integrasi : {df_integrated.shape[0]}")

print("\nContoh Data Setelah Integrasi:")
display(
    df_integrated[
        ["Username", "Paket Internet", "Wilayah (Mapping Alamat)",
         "Wilayah Pemasaran", "Kecepatan", "Harga (Rp)"]
    ].head(3)
)

3.3.3.4  DATA INTEGRATION (DATA MAPPING + INNER JOIN)


[integrate_catalog] Data Mapping Alamat: 105 baris tidak cocok kata kunci manapun, 68 baris hasil mapping BEDA dengan wilayah resmi paket (nama tempat ambigu lintas wilayah). Kolom 'Wilayah Pemasaran' (dari katalog) tetap dipakai sebagai sumber kebenaran untuk filter rekomendasi.

Jumlah Baris sebelum integrasi : 643
Jumlah Baris setelah integrasi : 619

Contoh Data Setelah Integrasi:


,Username,Paket Internet,Wilayah (Mapping Alamat),Wilayah Pemasaran,Kecepatan,Harga (Rp)
0,DANAN,WARGANET Lite 150,"PWS, Sudirman, Tigaraksa","Pinang, Pabuaran, Katomas, Leuwihalu",10.0,150000
1,ELFA,WARGANET Lite 100,"PWS, Sudirman, Tigaraksa","Pinang, Pabuaran, Katomas, Leuwihalu",4.0,100000
2,Pbr@agustobing.tbb,ACSPINANG 330,"PWS, Sudirman, Tigaraksa","Pinang, Pabuaran, Katomas, Leuwihalu",50.0,330000


### 3.3.3.5 Data Transformation (Min-Max Normalization)

In [8]:
print("="*70)
print("3.3.3.5  DATA TRANSFORMATION (MIN-MAX NORMALIZATION)")
print("="*70)

print("Sebelum normalisasi:")
display(df_integrated[["Total Online (detik)", "Total Kuota (bytes)"]].describe().loc[["min", "max"]])

df_scaled = dp.transform_features(df_integrated)

print("\nSetelah normalisasi (skala 0-1):")
display(df_scaled[["Total Online (detik)", "Total Kuota (bytes)"]].describe().loc[["min", "max"]])

3.3.3.5  DATA TRANSFORMATION (MIN-MAX NORMALIZATION)
Sebelum normalisasi:


,Total Online (detik),Total Kuota (bytes)
min,0.0000e+00,0.0000e+00
max,1.4634e+07,1.5200e+13



Setelah normalisasi (skala 0-1):


,Total Online (detik),Total Kuota (bytes)
min,0.0,0.0
max,1.0,1.0


### 3.3.3.6 Pembentukan Rating Implicit (Data Construction)

In [9]:
print("="*70)
print("3.3.3.6  PEMBENTUKAN RATING IMPLICIT (DATA CONSTRUCTION)")
print("="*70)

df_transformed = dp.construct_implicit_rating(df_scaled)

print("Contoh Implicit Rating:")
display(
    df_transformed[
        ["Username", "Paket Internet", "Total Online (detik)",
         "Total Kuota (bytes)", "Implicit Rating"]
    ].head(3)
)

3.3.3.6  PEMBENTUKAN RATING IMPLICIT (DATA CONSTRUCTION)
Contoh Implicit Rating:


,Username,Paket Internet,Total Online (detik),Total Kuota (bytes),Implicit Rating
0,DANAN,WARGANET Lite 150,0.5117,0.0265,0.2691
1,ELFA,WARGANET Lite 100,0.0000,0.0000,0.0000
2,Pbr@agustobing.tbb,ACSPINANG 330,0.4359,0.1382,0.2870


### Output Akhir Data Preparation

In [10]:
print("="*70)
print("OUTPUT AKHIR DATA PREPARATION")
print("="*70)

final_cols = [
    "Username", "Paket Internet", "Total Online (detik)", "Total Kuota (bytes)",
    "Nama Paket", "Wilayah Pemasaran", "Wilayah (Mapping Alamat)",
    "Kecepatan", "Harga (Rp)", "Implicit Rating",
]
df_transformed = df_transformed[final_cols]

print(f"Jumlah Baris : {df_transformed.shape[0]}")
print(f"Jumlah Kolom : {df_transformed.shape[1]}")

print("\nContoh Dataset Analitik Final:")
display(df_transformed.head(3))

df_transformed.to_csv("D_transformed.csv", index=False)
print("\nD_transformed.csv berhasil diregenerate.")

OUTPUT AKHIR DATA PREPARATION
Jumlah Baris : 619
Jumlah Kolom : 10

Contoh Dataset Analitik Final:


,Username,Paket Internet,Total Online (detik),Total Kuota (bytes),Nama Paket,Wilayah Pemasaran,Wilayah (Mapping Alamat),Kecepatan,Harga (Rp),Implicit Rating
0,DANAN,WARGANET Lite 150,0.5117,0.0265,WARGANET Lite 150,"Pinang, Pabuaran, Katomas, Leuwihalu","PWS, Sudirman, Tigaraksa",10.0,150000,0.2691
1,ELFA,WARGANET Lite 100,0.0000,0.0000,WARGANET Lite 100,"Pinang, Pabuaran, Katomas, Leuwihalu","PWS, Sudirman, Tigaraksa",4.0,100000,0.0000
2,Pbr@agustobing.tbb,ACSPINANG 330,0.4359,0.1382,ACSPINANG 330,"Pinang, Pabuaran, Katomas, Leuwihalu","PWS, Sudirman, Tigaraksa",50.0,330000,0.2870



D_transformed.csv berhasil diregenerate.


## 3.3.4 Modeling

### 3.3.4.1 Pembagian Data (Data Splitting 80:20)

In [11]:
print("="*70)
print("3.3.4.1  PEMBAGIAN DATA (DATA SPLITTING 80:20)")
print("="*70)

train_df, test_df = rec.split_train_test(df_transformed)

print(f"Total data analitik : {len(df_transformed)}")
print(f"Data latih (80%)    : {len(train_df)}")
print(f"Data uji (20%)      : {len(test_df)}")

3.3.4.1  PEMBAGIAN DATA (DATA SPLITTING 80:20)
Total data analitik : 619
Data latih (80%)    : 495
Data uji (20%)      : 124


### 3.3.4.2 Pembentukan User-Item Matrix

In [12]:
print("="*70)
print("3.3.4.2  PEMBENTUKAN USER-ITEM MATRIX")
print("="*70)

matrix = rec.build_user_item_matrix(train_df)

print(f"Ukuran Matrix : {matrix.shape[0]} pelanggan x {matrix.shape[1]} paket")
print("\nContoh Matrix (3 pelanggan pertama, 5 kolom pertama):")
display(matrix.iloc[:3, :5])

3.3.4.2  PEMBENTUKAN USER-ITEM MATRIX
Ukuran Matrix : 495 pelanggan x 20 paket

Contoh Matrix (3 pelanggan pertama, 5 kolom pertama):


Paket Internet,ACS CONNECT 185,ACS Home 165,ACS Lite 100,ACS Lite 130,ACS Lite 150
Username,,,,,
DANAN,0.0,0.0,0.0,0.0,0.0
ELFA,0.0,0.0,0.0,0.0,0.0
Sdr@mrizalfahmi,0.0,0.0,0.0,0.0,0.0


### 3.3.4.3 Perhitungan Jarak Kemiripan (Cosine Similarity)

In [13]:
print("="*70)
print("3.3.4.3  PERHITUNGAN JARAK KEMIRIPAN (COSINE SIMILARITY)")
print("="*70)

similarity = rec.compute_user_similarity(matrix)

sample_user = next(
    u for u in matrix.index
    if not rec.predict_scores(u, matrix, similarity, top_k=10).empty
)

print(f"Contoh kemiripan pelanggan '{sample_user}' terhadap 5 pelanggan lain:")
display(similarity[sample_user].sort_values(ascending=False).head(6))

3.3.4.3  PERHITUNGAN JARAK KEMIRIPAN (COSINE SIMILARITY)
Contoh kemiripan pelanggan 'Sdr@putriida' terhadap 5 pelanggan lain:


Username
putri@pws         1.0
kevinbela@pws     1.0
pws@lio_juleha    1.0
angga             1.0
pws@lio_suliyo    1.0
nanioman@pws      1.0
Name: Sdr@putriida, dtype: float64

### 3.3.4.4 Prediksi Skor (Weighted Sum)

In [14]:
print("="*70)
print("3.3.4.4  PREDIKSI SKOR (WEIGHTED SUM)")
print("="*70)

predicted = rec.predict_scores(sample_user, matrix, similarity, top_k=10)

print(f"Contoh prediksi skor untuk pelanggan '{sample_user}':")
if predicted.empty:
    print("(tidak ada kandidat -- tidak ada tetangga dengan paket berbeda)")
else:
    display(predicted.sort_values(ascending=False).head(10))

3.3.4.4  PREDIKSI SKOR (WEIGHTED SUM)
Contoh prediksi skor untuk pelanggan 'Sdr@putriida':


Paket Internet
ACS Lite 100    0.2382
dtype: float64

### 3.3.4.5 Pembangkitan Rekomendasi (Top-N Recommendation)

In [15]:
print("="*70)
print("3.3.4.5  PEMBANGKITAN REKOMENDASI (TOP-N RECOMMENDATION)")
print("="*70)

recommendations, neighbor_table = rec.recommend_existing_customer(
    sample_user, train_df, matrix, similarity, top_k=10
)

print(f"Rekomendasi Top-N untuk pelanggan '{sample_user}':")
display(recommendations.drop(columns=["Sumber"]) if not recommendations.empty else recommendations)

if not recommendations.empty and (recommendations["Sumber"] != "Collaborative Filtering").all():
    print(
        "\nSumber: Fallback Hybrid (BUKAN Rule-Based Filtering 3.3.4.6 -- itu berdiri sendiri, "
        "khusus Cold Start pelanggan BARU). Catatan: skor Weighted Sum pada 3.3.4.4 di atas ternyata "
        "untuk paket yang SUDAH dipakai pelanggan ini sendiri (kebetulan dipakai juga oleh "
        "tetangganya), sehingga dieksklusi oleh syarat kepemilikan (a) pada Top-N Recommendation. "
        "Sistem otomatis beralih ke fallback internal (skor kedekatan simetris ke harga & kecepatan "
        "paket yang sedang dipakai pelanggan) sesuai arsitektur Hybrid -- ini konsisten terjadi di "
        "hampir seluruh pelanggan pada dataset ini karena mayoritas hanya tercatat memakai satu paket."
    )

3.3.4.5  PEMBANGKITAN REKOMENDASI (TOP-N RECOMMENDATION)
Rekomendasi Top-N untuk pelanggan 'Sdr@putriida':


,Paket Internet,Skor Prediksi,Tingkat Kecocokan (%),Wilayah Pemasaran,Kecepatan,Harga (Rp)
0,ACS Pws 100 3Mbps,0.9897,98.97,"PWS, Sudirman, Tigaraksa",3.0,100000
1,ACS Lite 80,0.9488,94.88,"PWS, Sudirman, Tigaraksa",2.5,80000
2,ACS Lite 130,0.9258,92.58,"PWS, Sudirman, Tigaraksa",6.0,130000
3,ACS Pws 140 5Mbps,0.9183,91.83,"PWS, Sudirman, Tigaraksa",5.0,140000
4,ACS Lite 50,0.8849,88.49,"PWS, Sudirman, Tigaraksa",1.5,50000
5,ACS Pws 150 7Mbps,0.8798,87.98,"PWS, Sudirman, Tigaraksa",7.0,150000
6,ACS Lite 150,0.8489,84.89,"PWS, Sudirman, Tigaraksa",10.0,150000
7,ACS Home 165,0.8221,82.21,"PWS, Sudirman, Tigaraksa",10.0,165000
8,ACS Lite 165,0.7705,77.05,"PWS, Sudirman, Tigaraksa",15.0,165000
9,ACS CONNECT 185,0.6833,68.33,"PWS, Sudirman, Tigaraksa",20.0,185000



Sumber: Fallback Hybrid (BUKAN Rule-Based Filtering 3.3.4.6 -- itu berdiri sendiri, khusus Cold Start pelanggan BARU). Catatan: skor Weighted Sum pada 3.3.4.4 di atas ternyata untuk paket yang SUDAH dipakai pelanggan ini sendiri (kebetulan dipakai juga oleh tetangganya), sehingga dieksklusi oleh syarat kepemilikan (a) pada Top-N Recommendation. Sistem otomatis beralih ke fallback internal (skor kedekatan simetris ke harga & kecepatan paket yang sedang dipakai pelanggan) sesuai arsitektur Hybrid -- ini konsisten terjadi di hampir seluruh pelanggan pada dataset ini karena mayoritas hanya tercatat memakai satu paket.


### Demonstrasi Fallback Hybrid (CF Tidak Punya Tetangga Lintas-Paket)
Ini **BUKAN** Rule-Based Filtering (3.3.4.6) -- itu algoritma yang berdiri sendiri, khusus Cold Start pelanggan BARU (input Budget & Kecepatan Minimal lewat form). Ini demonstrasi cabang LAIN dari `recommend_existing_customer()` (3.3.4.5) yang aktif saat CF gagal total mendapat sinyal.

Contoh di bawah sengaja pakai pelanggan **'DANAN'**, yang `predict_scores()`-nya **kosong total** (similarity ke SEMUA pelanggan lain = 0, tidak ada satupun tetangga dengan paket berbeda) -- kasus paling jelas untuk membuktikan kapan & kenapa fallback ini aktif. Formulanya (kedekatan SIMETRIS ke harga & kecepatan paket yang SEDANG dipakai) beda dari Rule-Based Filtering 3.3.4.6 (threshold Harga<=Budget AND Kecepatan>=MinSpeed) -- lihat `recommender.py` fungsi `recommend_existing_customer()` blok fallback.

In [16]:
print("="*70)
print("DEMONSTRASI FALLBACK HYBRID (CF TIDAK PUNYA TETANGGA LINTAS-PAKET)")
print("="*70)

demo_fallback_user = "DANAN"
row_demo = train_df[train_df["Username"] == demo_fallback_user].iloc[0]

print(f"Pelanggan contoh   : '{demo_fallback_user}'")
print(f"Paket saat ini     : {row_demo['Paket Internet']}")
print(f"Wilayah            : {row_demo['Wilayah Pemasaran']}")
print(f"Harga saat ini     : Rp{row_demo['Harga (Rp)']:,.0f}")
print(f"Kecepatan saat ini : {row_demo['Kecepatan']} Mbps")

print(f"\nSimilarity '{demo_fallback_user}' ke 5 pelanggan lain (tertinggi):")
display(similarity[demo_fallback_user].sort_values(ascending=False).head(5))

predicted_demo = rec.predict_scores(demo_fallback_user, matrix, similarity, top_k=10)
status_predict = "KOSONG (tidak ada tetangga similarity > 0 dgn paket berbeda)" if predicted_demo.empty else "ada kandidat"
print(f"\npredict_scores() -> {status_predict}")

recs_fallback, _ = rec.recommend_existing_customer(demo_fallback_user, train_df, matrix, similarity, top_k=10)
print(f"\nHasil Fallback Hybrid untuk '{demo_fallback_user}':")
display(recs_fallback)

DEMONSTRASI FALLBACK HYBRID (CF TIDAK PUNYA TETANGGA LINTAS-PAKET)
Pelanggan contoh   : 'DANAN'
Paket saat ini     : WARGANET Lite 150
Wilayah            : Pinang, Pabuaran, Katomas, Leuwihalu
Harga saat ini     : Rp150,000
Kecepatan saat ini : 10.0 Mbps

Similarity 'DANAN' ke 5 pelanggan lain (tertinggi):


Username
DANAN               1.0
ELFA                0.0
Sdr@mrizalfahmi     0.0
Sdr@putriida        0.0
abyan3madura@pws    0.0
Name: DANAN, dtype: float64


predict_scores() -> KOSONG (tidak ada tetangga similarity > 0 dgn paket berbeda)



Hasil Fallback Hybrid untuk 'DANAN':


,Paket Internet,Skor Prediksi,Tingkat Kecocokan (%),Wilayah Pemasaran,Kecepatan,Harga (Rp),Sumber
0,ACSPINANG 165,0.9674,96.74,"Pinang, Pabuaran, Katomas, Leuwihalu",10.0,165000,Fallback Hybrid (Kedekatan Profil Paket — CF t...
1,WARGANET Lite 100,0.8261,82.61,"Pinang, Pabuaran, Katomas, Leuwihalu",4.0,100000,Fallback Hybrid (Kedekatan Profil Paket — CF t...
2,ACSPINANG 220,0.7391,73.91,"Pinang, Pabuaran, Katomas, Leuwihalu",20.0,220000,Fallback Hybrid (Kedekatan Profil Paket — CF t...
3,ACSPINANG 250,0.5652,56.52,"Pinang, Pabuaran, Katomas, Leuwihalu",30.0,250000,Fallback Hybrid (Kedekatan Profil Paket — CF t...
4,ACSPINANG 330,0.1739,17.39,"Pinang, Pabuaran, Katomas, Leuwihalu",50.0,330000,Fallback Hybrid (Kedekatan Profil Paket — CF t...


### 3.3.4.6 Penanganan Cold Start Problem (Rule-Based Filtering)

In [17]:
print("="*70)
print("3.3.4.6  PENANGANAN COLD START PROBLEM (RULE-BASED FILTERING)")
print("="*70)

catalog_demo = train_df.drop_duplicates(subset=["Paket Internet"])[
    ["Paket Internet", "Wilayah Pemasaran", "Kecepatan", "Harga (Rp)"]
]
demo_wilayah = "PWS, Sudirman, Tigaraksa"
demo_budget = 200000
demo_speed = 10

print(f"Contoh calon pelanggan baru: Budget=Rp{demo_budget:,}, "
      f"MinSpeed={demo_speed}Mbps, Wilayah='{demo_wilayah}'")

new_customer_reco = rec.recommend_new_customer(catalog_demo, demo_wilayah, demo_budget, demo_speed)
display(new_customer_reco)

3.3.4.6  PENANGANAN COLD START PROBLEM (RULE-BASED FILTERING)
Contoh calon pelanggan baru: Budget=Rp200,000, MinSpeed=10Mbps, Wilayah='PWS, Sudirman, Tigaraksa'


,Paket Internet,Wilayah Pemasaran,Kecepatan,Harga (Rp),Skor Budget,Skor Kecepatan,Skor Total
415,ACS CONNECT 185,"PWS, Sudirman, Tigaraksa",20.0,185000,0.925,1.00,0.97
531,ACS Lite 165,"PWS, Sudirman, Tigaraksa",15.0,165000,0.825,0.75,0.78
153,ACS Home 165,"PWS, Sudirman, Tigaraksa",10.0,165000,0.825,0.50,0.63
286,ACS Lite 150,"PWS, Sudirman, Tigaraksa",10.0,150000,0.750,0.50,0.60


## 3.3.5 Evaluasi

### Mean Absolute Error (MAE)

In [18]:
print("="*70)
print("3.3.5  EVALUASI -- MEAN ABSOLUTE ERROR (MAE)")
print("="*70)

hasil_mae = rec.evaluate_mae(train_df, test_df)

print(f"MAE  : {hasil_mae['mae']:.4f}")
print(f"Jumlah data uji : {hasil_mae['n_test']}")

print("\nContoh Detail Galat Prediksi (MAE):")
display(hasil_mae["detail"].head(5))

3.3.5  EVALUASI -- MEAN ABSOLUTE ERROR (MAE)


MAE  : 0.0611
Jumlah data uji : 124

Contoh Detail Galat Prediksi (MAE):


,Username,Paket Internet,Implicit Rating Aktual,Implicit Rating Prediksi,Galat Absolut
0,ktm@sefrihardiansyah,ACSPINANG 165,0.2690,0.2347,0.0343
1,trv2@imamsugianto,ACSPINANG 165,0.0067,0.2016,0.1949
2,mutiara@belimbing,ACS Lite 150,0.2479,0.2286,0.0194
3,sdr@a1fitria09,ACS Lite 220,0.2672,0.2815,0.0143
4,pbr@karlesjamu,ACSPINANG 165,0.1890,0.2347,0.0458


### Root Mean Square Error (RMSE)

In [19]:
print("="*70)
print("3.3.5  EVALUASI -- ROOT MEAN SQUARE ERROR (RMSE)")
print("="*70)

hasil_rmse = rec.evaluate_rmse(train_df, test_df)

print(f"RMSE : {hasil_rmse['rmse']:.4f}")
print(f"Jumlah data uji : {hasil_rmse['n_test']}")

print("\nContoh Detail Galat Prediksi (RMSE):")
display(hasil_rmse["detail"].head(5))

3.3.5  EVALUASI -- ROOT MEAN SQUARE ERROR (RMSE)


RMSE : 0.0956
Jumlah data uji : 124

Contoh Detail Galat Prediksi (RMSE):


,Username,Paket Internet,Implicit Rating Aktual,Implicit Rating Prediksi,Kuadrat Galat
0,ktm@sefrihardiansyah,ACSPINANG 165,0.2690,0.2347,0.0012
1,trv2@imamsugianto,ACSPINANG 165,0.0067,0.2016,0.0380
2,mutiara@belimbing,ACS Lite 150,0.2479,0.2286,0.0004
3,sdr@a1fitria09,ACS Lite 220,0.2672,0.2815,0.0002
4,pbr@karlesjamu,ACSPINANG 165,0.1890,0.2347,0.0021


## Validasi Sistem Tambahan — Uji Reliabilitas Cronbach's Alpha
**Arahan pembimbing**: dilihat dari paket yang SEDANG dipakai pelanggan, lalu sistem menghitung pelanggan itu cocok pakai layanan yang mana -- persis seperti tabel "Validasi Sistem" (Username | Paket Saat Ini | Status | Daftar Rekomendasi) yang sudah berjalan di `app.py`.

**Item** = skor **Tingkat Kecocokan (%)** di tiap **peringkat** rekomendasi (Peringkat 1, Peringkat 2, ..., Peringkat N) -- bukan indikator cocok/tidak-cocok biner. **Responden** = tiap pelanggan di data uji. Cronbach's Alpha mengukur apakah pola skor kecocokan itu **konsisten** antar pelanggan (mis. peringkat 1 selalu tertinggi, menurun bertahap ke peringkat berikutnya, dan pola itu stabil di seluruh pelanggan).

**Catatan metodologis (penting)**: Cronbach's Alpha di sini mengukur **konsistensi internal** pola skor Top-N -- BUKAN akurasi tebakan paket. Paket yang sedang dipakai pelanggan ("Paket Saat Ini") **tidak** menjadi komponen perhitungan Alpha; kolom itu murni label referensi. Cek "apakah paket saat ini muncul di Top-N kandidat" dilakukan **terpisah** lewat kolom "Paket Saat Ini di Peringkat" di bawah, bukan bagian dari rumus Alpha.

`rec.evaluate_rank_confidence()` mereplikasi PERSIS logika `run_full_validation()` di `app.py` (tab "Validasi Sistem" yang sudah berjalan sekarang: jarak Euclidean, neighbor_pool=30) -- validasi yang sudah ada TIDAK diubah atau dihapus, ini murni tambahan.

In [20]:
print("="*70)
print("VALIDASI SISTEM TAMBAHAN -- CRONBACH'S ALPHA")
print("="*70)

TOP_N = 5
rank_df = rec.evaluate_rank_confidence(test_df, train_df, top_n=TOP_N)
rank_cols = [f"Peringkat {i} (%)" for i in range(1, TOP_N + 1)]

print(f"Jumlah pelanggan di data uji           : {len(rank_df)}")
print("\nContoh Skor Tingkat Kecocokan per Peringkat:")
display(rank_df.head(6))

VALIDASI SISTEM TAMBAHAN -- CRONBACH'S ALPHA


Jumlah pelanggan di data uji           : 124

Contoh Skor Tingkat Kecocokan per Peringkat:


,Username,Paket Saat Ini,Peringkat 1 (%),Peringkat 2 (%),Peringkat 3 (%),Peringkat 4 (%),Peringkat 5 (%),Paket Saat Ini di Peringkat
0,ktm@sefrihardiansyah,ACSPINANG 165,99.57,99.50,99.06,99.05,98.96,NaN
1,trv2@imamsugianto,ACSPINANG 165,99.22,99.17,98.98,98.93,98.93,2.0
2,mutiara@belimbing,ACS Lite 150,99.87,99.65,99.65,99.52,99.49,3.0
3,sdr@a1fitria09,ACS Lite 220,99.64,99.29,99.02,99.00,99.00,NaN
4,pbr@karlesjamu,ACSPINANG 165,97.48,96.39,95.93,95.67,95.40,2.0
5,trv@suparmanto.d1no12,ACSPINANG 165,99.95,99.80,99.63,99.57,99.57,3.0


In [21]:
print("="*70)
print("PAKET SAAT INI DIKENALI SISTEM? (DI LUAR PERHITUNGAN ALPHA)")
print("="*70)

n_dikenali = rank_df["Paket Saat Ini di Peringkat"].notna().sum()
n_total_rank = len(rank_df)

print(
    f"Paket yang SEDANG dipakai pelanggan muncul di Top-{TOP_N} kandidat "
    f"rekomendasi mesin: {n_dikenali}/{n_total_rank} pelanggan "
    f"({n_dikenali / n_total_rank * 100:.1f}%)"
)

print()
print("Contoh (6 pelanggan pertama):")
display(rank_df[["Username", "Paket Saat Ini", "Paket Saat Ini di Peringkat"]].head(6))

PAKET SAAT INI DIKENALI SISTEM? (DI LUAR PERHITUNGAN ALPHA)
Paket yang SEDANG dipakai pelanggan muncul di Top-5 kandidat rekomendasi mesin: 92/124 pelanggan (74.2%)

Contoh (6 pelanggan pertama):


,Username,Paket Saat Ini,Paket Saat Ini di Peringkat
0,ktm@sefrihardiansyah,ACSPINANG 165,NaN
1,trv2@imamsugianto,ACSPINANG 165,2.0
2,mutiara@belimbing,ACS Lite 150,3.0
3,sdr@a1fitria09,ACS Lite 220,NaN
4,pbr@karlesjamu,ACSPINANG 165,2.0
5,trv@suparmanto.d1no12,ACSPINANG 165,3.0


In [22]:
# Cronbach's Alpha butuh data LENGKAP semua item per responden -- pelanggan
# yang kandidat paketnya kurang dari TOP_N (rekomendasinya tidak penuh)
# di-drop dulu.
rank_complete = rank_df.dropna(subset=rank_cols)
print(f"Pelanggan dengan {TOP_N} peringkat lengkap : {len(rank_complete)} dari {len(rank_df)}")

alpha = rec.compute_cronbach_alpha(rank_complete[rank_cols])
label = rec.interpret_cronbach_alpha(alpha)

print(f"\nCronbach's Alpha : {alpha:.4f}")
print(f"Interpretasi     : {label}")
print(f"Jumlah item (k)  : {len(rank_cols)}")
print(f"Jumlah responden : {len(rank_complete)}")

Pelanggan dengan 5 peringkat lengkap : 123 dari 124

Cronbach's Alpha : 0.9418
Interpretasi     : Sangat Baik (Excellent)
Jumlah item (k)  : 5
Jumlah responden : 123


## Perbandingan Metode Tambahan — Content-Based Filtering
**Arahan pembimbing**: membandingkan AKURASI dan HASIL REKOMENDASI antara **User-Based Collaborative Filtering** (metode utama di `app.py`, sesuai draft skripsi Bab III.3.4) dengan **Content-Based Filtering** (algoritma pembanding, kemiripan atribut paket).

Content-Based Filtering merekomendasikan paket berdasar kemiripan **atribut paket** (Kecepatan, Harga) terhadap paket yang sedang dipakai pelanggan -- bukan kemiripan antar pelanggan seperti Collaborative Filtering. Karena tidak butuh overlap pemakaian antar pelanggan sama sekali (murni katalog), metode ini tidak terpengaruh masalah sparsitas data yang sama.

Sel-sel di bawah memanggil fungsi ASLI dari `content_based.py` (`import content_based as cb`) -- sama seperti `app_content_based.py` -- sehingga hasilnya dijamin identik dengan aplikasi Streamlit perbandingan metode.

### Pembentukan Profil Item & Kemiripan Antar Paket (Content-Based)

In [23]:
print("="*70)
print("PERBANDINGAN METODE -- PEMBENTUKAN PROFIL ITEM (CONTENT-BASED)")
print("="*70)

import content_based as cb

katalog_cb = pd.read_csv("Jenis_Layanan.csv")
katalog_cb.columns = katalog_cb.columns.str.strip()
katalog_cb["Kecepatan"] = (
    katalog_cb["Kecepatan"].astype(str).str.replace(" Mbps", "", regex=False).str.strip().astype(float)
)
katalog_cb["Harga (Rp)"] = pd.to_numeric(katalog_cb["Harga (Rp)"], errors="coerce")

item_profile = cb.build_item_profile(katalog_cb)
item_similarity = cb.compute_item_similarity(item_profile)

print(f"Jumlah paket pada profil item : {len(item_profile)}")
print("\nContoh Profil Item (Kecepatan & Harga setelah Min-Max):")
display(item_profile.head(5))

print("\nContoh Kemiripan Antar Paket (5x5 pertama):")
display(item_similarity.iloc[:5, :5])

PERBANDINGAN METODE -- PEMBENTUKAN PROFIL ITEM (CONTENT-BASED)
Jumlah paket pada profil item : 22

Contoh Profil Item (Kecepatan & Harga setelah Min-Max):


,Kecepatan,Harga (Rp)
Nama Paket,,
ACS Pws 100 3Mbps,0.0309,0.1786
ACS Pws 140 5Mbps,0.0722,0.3214
ACS Pws 150 7Mbps,0.1134,0.3571
ACS Home 165,0.1753,0.4107
ACS CONNECT 185,0.3814,0.4821



Contoh Kemiripan Antar Paket (5x5 pertama):


Nama Paket,ACS Pws 100 3Mbps,ACS Pws 140 5Mbps,ACS Pws 150 7Mbps,ACS Home 165,ACS CONNECT 185
Nama Paket,,,,,
ACS Pws 100 3Mbps,1.0000,0.8949,0.8609,0.8067,0.6721
ACS Pws 140 5Mbps,0.8949,1.0000,0.9614,0.9036,0.7535
ACS Pws 150 7Mbps,0.8609,0.9614,1.0000,0.9421,0.7909
ACS Home 165,0.8067,0.9036,0.9421,1.0000,0.8457
ACS CONNECT 185,0.6721,0.7535,0.7909,0.8457,1.0000


### Rekomendasi: Perbandingan CF vs Content-Based untuk 1 Pelanggan

In [24]:
print("="*70)
print("PERBANDINGAN METODE -- REKOMENDASI (CF vs CONTENT-BASED)")
print("="*70)

full_matrix = rec.build_user_item_matrix(df_transformed)
full_similarity = rec.compute_user_similarity(full_matrix)

TOP_N_COMPARE = 5
recs_cf, _ = rec.recommend_existing_customer(sample_user, df_transformed, full_matrix, full_similarity, top_k=10)
recs_cf = recs_cf.head(TOP_N_COMPARE)
recs_cb, current_package_cb = cb.recommend_content_based(sample_user, df_transformed, item_similarity, top_n=TOP_N_COMPARE)

print(f"Pelanggan: '{sample_user}' (paket saat ini: {current_package_cb})")

print("\nRekomendasi -- User-Based Collaborative Filtering:")
display(recs_cf[["Paket Internet", "Tingkat Kecocokan (%)", "Harga (Rp)"]] if not recs_cf.empty else recs_cf)

print("\nRekomendasi -- Content-Based Filtering:")
display(recs_cb[["Paket Internet", "Tingkat Kecocokan (%)", "Harga (Rp)"]] if not recs_cb.empty else recs_cb)

overlap = (
    (set(recs_cf["Paket Internet"]) if not recs_cf.empty else set())
    & (set(recs_cb["Paket Internet"]) if not recs_cb.empty else set())
)
print(f"\nPaket yang muncul di KEDUA metode: {', '.join(overlap) if overlap else '(tidak ada yang sama)'}")

PERBANDINGAN METODE -- REKOMENDASI (CF vs CONTENT-BASED)


Pelanggan: 'Sdr@putriida' (paket saat ini: ACS Lite 100)

Rekomendasi -- User-Based Collaborative Filtering:


,Paket Internet,Tingkat Kecocokan (%),Harga (Rp)
0,ACS Pws 100 3Mbps,98.97,100000
1,ACS Lite 80,94.88,80000
2,ACS Lite 100 75,94.50,75000
3,ACS Lite 130,92.58,130000
4,ACS Pws 140 5Mbps,91.83,140000



Rekomendasi -- Content-Based Filtering:


,Paket Internet,Tingkat Kecocokan (%),Harga (Rp)
0,ACS Pws 100 3Mbps,98.54,100000
1,ACS Lite 80,94.50,80000
2,ACS Lite 100 75,93.52,75000
3,ACS Lite 130,91.88,130000
4,ACS Pws 140 5Mbps,89.79,140000



Paket yang muncul di KEDUA metode: ACS Lite 100 75, ACS Lite 80, ACS Lite 130, ACS Pws 140 5Mbps, ACS Pws 100 3Mbps


### Evaluasi MAE & RMSE: Perbandingan Langsung

In [25]:
print("="*70)
print("PERBANDINGAN METODE -- EVALUASI MAE & RMSE")
print("="*70)

eval_cf = rec.evaluate_mae_rmse(train_df, test_df, top_k=10)
eval_cb = cb.evaluate_mae_rmse(train_df, test_df, item_similarity)

compare_df = pd.DataFrame(
    {
        "Metrik": ["MAE", "RMSE"],
        "Collaborative Filtering": [eval_cf["mae"], eval_cf["rmse"]],
        "Content-Based Filtering": [eval_cb["mae"], eval_cb["rmse"]],
    }
)

print(f"Data Latih : {len(train_df)}")
print(f"Data Uji   : {eval_cf['n_test']} (identik untuk kedua metode)")
print("\nTabel Perbandingan MAE & RMSE:")
display(compare_df)

PERBANDINGAN METODE -- EVALUASI MAE & RMSE


Data Latih : 495
Data Uji   : 124 (identik untuk kedua metode)

Tabel Perbandingan MAE & RMSE:


,Metrik,Collaborative Filtering,Content-Based Filtering
0,MAE,0.0611,0.0662
1,RMSE,0.0956,0.0967


## 3.3.6 Deployment
Model yang sudah divalidasi di atas diimplementasikan ke aplikasi web Streamlit (`app.py` / `app_final.py`). Kedua file itu memanggil fungsi **yang sama persis** dari `data_preparation.py` dan `recommender.py` yang dipakai di notebook ini (`import recommender as rec`, `import data_preparation as dp`) -- jadi apa pun yang terlihat di layar aplikasi dijamin identik dengan hasil komputasi di atas.

### Implementasi Skenario Pelanggan Lama (Collaborative Filtering)

In [26]:
print("="*70)
print("IMPLEMENTASI: TAB 'PELANGGAN LAMA' DI APP.PY")
print("="*70)
print(
    f"app.py memanggil rec.recommend_existing_customer() -- fungsi yang SAMA "
    f"dipakai di 3.3.4.5 di atas. Kalau staf mencari username "
    f"'{sample_user}' pada tab 'Pelanggan Lama', tabel Top-N Recommendation "
    f"yang tampil di layar akan IDENTIK dengan tabel berikut:"
)
display(recommendations.drop(columns=["Sumber"]) if not recommendations.empty else recommendations)

IMPLEMENTASI: TAB 'PELANGGAN LAMA' DI APP.PY
app.py memanggil rec.recommend_existing_customer() -- fungsi yang SAMA dipakai di 3.3.4.5 di atas. Kalau staf mencari username 'Sdr@putriida' pada tab 'Pelanggan Lama', tabel Top-N Recommendation yang tampil di layar akan IDENTIK dengan tabel berikut:


,Paket Internet,Skor Prediksi,Tingkat Kecocokan (%),Wilayah Pemasaran,Kecepatan,Harga (Rp)
0,ACS Pws 100 3Mbps,0.9897,98.97,"PWS, Sudirman, Tigaraksa",3.0,100000
1,ACS Lite 80,0.9488,94.88,"PWS, Sudirman, Tigaraksa",2.5,80000
2,ACS Lite 130,0.9258,92.58,"PWS, Sudirman, Tigaraksa",6.0,130000
3,ACS Pws 140 5Mbps,0.9183,91.83,"PWS, Sudirman, Tigaraksa",5.0,140000
4,ACS Lite 50,0.8849,88.49,"PWS, Sudirman, Tigaraksa",1.5,50000
5,ACS Pws 150 7Mbps,0.8798,87.98,"PWS, Sudirman, Tigaraksa",7.0,150000
6,ACS Lite 150,0.8489,84.89,"PWS, Sudirman, Tigaraksa",10.0,150000
7,ACS Home 165,0.8221,82.21,"PWS, Sudirman, Tigaraksa",10.0,165000
8,ACS Lite 165,0.7705,77.05,"PWS, Sudirman, Tigaraksa",15.0,165000
9,ACS CONNECT 185,0.6833,68.33,"PWS, Sudirman, Tigaraksa",20.0,185000


### Implementasi Skenario Pelanggan Baru (Rule-Based Filtering)

In [27]:
print("="*70)
print("IMPLEMENTASI: TAB 'PELANGGAN BARU' DI APP.PY")
print("="*70)
print(
    f"app.py memanggil rec.recommend_new_customer() -- fungsi yang SAMA "
    f"dipakai di 3.3.4.6 di atas. Kalau staf mengisi form 'Pelanggan Baru' "
    f"dengan Budget=Rp{demo_budget:,}, Kecepatan Minimal={demo_speed}Mbps, "
    f"Wilayah='{demo_wilayah}', hasil rekomendasi yang tampil di layar akan "
    f"IDENTIK dengan tabel berikut:"
)
display(new_customer_reco)

IMPLEMENTASI: TAB 'PELANGGAN BARU' DI APP.PY
app.py memanggil rec.recommend_new_customer() -- fungsi yang SAMA dipakai di 3.3.4.6 di atas. Kalau staf mengisi form 'Pelanggan Baru' dengan Budget=Rp200,000, Kecepatan Minimal=10Mbps, Wilayah='PWS, Sudirman, Tigaraksa', hasil rekomendasi yang tampil di layar akan IDENTIK dengan tabel berikut:


,Paket Internet,Wilayah Pemasaran,Kecepatan,Harga (Rp),Skor Budget,Skor Kecepatan,Skor Total
415,ACS CONNECT 185,"PWS, Sudirman, Tigaraksa",20.0,185000,0.925,1.00,0.97
531,ACS Lite 165,"PWS, Sudirman, Tigaraksa",15.0,165000,0.825,0.75,0.78
153,ACS Home 165,"PWS, Sudirman, Tigaraksa",10.0,165000,0.825,0.50,0.63
286,ACS Lite 150,"PWS, Sudirman, Tigaraksa",10.0,150000,0.750,0.50,0.60


### Cara Menjalankan Aplikasi Streamlit

In [28]:
print("="*70)
print("3.3.6  CARA MENJALANKAN APLIKASI STREAMLIT")
print("="*70)

import json

with open(".claude/launch.json", encoding="utf-8") as f:
    launch_cfg = json.load(f)

for cfg in launch_cfg["configurations"]:
    cmd = " ".join([f'"{cfg["runtimeExecutable"]}"'] + [str(a) for a in cfg["runtimeArgs"]])
    print(f"\n{cfg['name']} -> http://localhost:{cfg['port']}")
    print(cmd)

3.3.6  CARA MENJALANKAN APLIKASI STREAMLIT

app -> http://localhost:8501
"C:\Users\THINKPAD\AppData\Local\Python\pythoncore-3.14-64\python.exe" -m streamlit run app.py --server.port 8501 --server.headless true

app_final -> http://localhost:8502
"C:\Users\THINKPAD\AppData\Local\Python\pythoncore-3.14-64\python.exe" -m streamlit run app_final.py --server.port 8502 --server.headless true

app_content_based -> http://localhost:8503
"C:\Users\THINKPAD\AppData\Local\Python\pythoncore-3.14-64\python.exe" -m streamlit run app_content_based.py --server.port 8503 --server.headless true
